# 🚀 Kaggle Setup Instructions

**Before running:**
1. ⚙️ **Enable GPU**: Settings → Accelerator → **GPU T4 x2**
2. ▶️ **Run All Cells**: Cell → Run All

**What happens:**
- Cells 1-2: Auto-detect Kaggle, install dependencies (~2 min)
- Cell 3: Download DeepSeek-Math-7B (~5-10 min, 13GB)
- Cell 4-8: Setup ReAct solver
- Cell 9: Process 10 questions (BATCH_SIZE=10)
- Cell 10: Show metrics

**To process all 198 questions:** Re-run cell 9 multiple times (auto-checkpoints after each problem)

# GPQA Diamond – ReAct Verification
- 198 PhD-level multiple-choice questions (Bio/Chem/Phys)
- **ReAct Method**: Use tools (Wikipedia, web search) to reason and verify answers
- **Output**: A/B/C/D only, full reasoning traces, metrics

In [ ]:
# Setup - Auto-detect environment
import os, sys, json, time, re, math
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# Detect environment
IS_COLAB = False
IS_KAGGLE = False
ENV_NAME = "Local"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    ENV_NAME = "Colab"
    BASE_PATH = Path('/content/drive/MyDrive/AGoT-ReAct/Math Performance')
except:
    # Check if Kaggle
    if os.path.exists('/kaggle/working'):
        IS_KAGGLE = True
        ENV_NAME = "Kaggle"
        BASE_PATH = Path('/kaggle/working')
    else:
        BASE_PATH = Path(r'f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance')

print(f"{ENV_NAME} Environment | {BASE_PATH}")
print(f"Base path exists: {BASE_PATH.exists()}")

In [ ]:
# Install dependencies
if IS_COLAB or IS_KAGGLE:
    os.system('pip install -q transformers torch accelerate datasets tqdm beautifulsoup4 requests')
    print("✓ Dependencies installed (Colab/Kaggle)")
else:
    os.system('pip install -q -r requirements.txt')
    print("✓ Dependencies installed from requirements.txt")

In [ ]:
rt pandas as pd
from tqdm import tqdm
import requests
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from dotenv import load_dotenv
    load_dotenv()
except:
    pass

# Paths & config
OUTPUT_DIR = BASE_PATH / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GPQA_OUTPUT_PATH = OUTPUT_DIR / 'gpqa_react_results.jsonl'
GPQA_TRACES_PATH = OUTPUT_DIR / 'gpqa_react_detailed_traces.jsonl'
GPQA_METRICS_PATH = OUTPUT_DIR / 'gpqa_metrics.json'
GPQA_CUMULATIVE_PATH = OUTPUT_DIR / 'gpqa_react_cumulative_metrics.json'
GPQA_CHECKPOINT_PATH = OUTPUT_DIR / 'gpqa_react_checkpoint.json'

# DeepSeek from HuggingFace (no API key needed)
MODEL_NAME = 'deepseek-ai/deepseek-math-7b-instruct'
BATCH_SIZE = 10
REACT_MAX_STEPS = 5

# Load model from HuggingFace
print(f"Loading {MODEL_NAME} from HuggingFace...")
print("This may take a few minutes on first run...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected - model will run on CPU (slower)")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    
    # Load with optimizations for better memory usage
    if device == "cuda":
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            trust_remote_code=True,
            torch_dtype=torch.float16,  # Use FP16 for faster inference
            device_map="auto",
            low_cpu_mem_usage=True
        )
    else:
        print("⚠️ Running on CPU - this will be slow. Consider using Google Colab for GPU access.")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            trust_remote_code=True,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True
        )
        model = model.to(device)
    
    model.eval()
    print(f"✓ Model loaded successfully on {device}")
    
except Exception as e:
    print(f"⚠️ Error loading model: {e}")
    print("Make sure you have enough disk space and RAM/VRAM")
    raise

print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Ready!")

In [ ]:
# Load GPQA Diamond
from datasets import load_dataset

print("Loading GPQA Diamond...")
gpqa_dataset = load_dataset("fingertap/GPQA-Diamond", split="test")
print(f"✓ Loaded {len(gpqa_dataset)} questions")
print(f"Fields: {gpqa_dataset.column_names}")
print(json.dumps({k: str(v)[:120] for k, v in gpqa_dataset[0].items()}, indent=2))

## External Tools for ReAct

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus

# ========================================
# DEPRECATED - Use Offline WikiEnv Instead
# ========================================
# This cell is kept for reference but NOT used by ReAct solver
# The actual tool execution happens in WikiEnv (Cell 9) which is OFFLINE
# WARNING: These methods require external API access - NOT used in this notebook

class ExternalToolExecutor:
    """DEPRECATED: Kept for reference only. Use WikiEnv for actual execution."""
    def __init__(self):
        self.search_history = []

    def search_wikipedia(self, entity: str) -> str:
        """NOT USED - See WikiEnv.search_local() instead"""
        # This would require external API access
        return f"[OFFLINE] Use WikiEnv.search[{entity}] instead"

    def search_web(self, query: str) -> str:
        """NOT USED - Offline knowledge base used instead"""
        return f"[OFFLINE] Uses local knowledge base"

    def lookup_in_text(self, keyword: str, context: str) -> str:
        """NOT USED - See WikiEnv.lookup[] instead"""
        if not context:
            return "No context."
        sentences = context.replace('\n', ' ').split('.')
        matches = [s.strip() for s in sentences if keyword.lower() in s.lower() and len(s.strip()) > 5]
        if matches:
            joined = '. '.join(matches[:2]) + '.'
            words = joined.split()
            return ' '.join(words[:120])
        return f"'{keyword}' not found."

external_tools = ExternalToolExecutor()
print("✓ External tools placeholder (ACTUAL: offline WikiEnv - NO external API calls)")

In [ ]:
import ast
import time
import gym
import numpy as np
import string
from collections import Counter
from bs4 import BeautifulSoup

# ========================================
# ReAct-Master: Complete Implementation
# (Offline - No External API Calls)
# ========================================

# ====================
# Utility Functions
# ====================

def clean_str(p):
    """Clean unicode and encoding issues."""
    try:
        return p.encode().decode("unicode-escape").encode("latin1").decode("utf-8")
    except:
        return p

def normalize_answer(s):
    """Normalize answer for evaluation (from ReAct-master)."""
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    
    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    """Calculate F1 score (from ReAct-master)."""
    normalized_prediction = normalize_answer(prediction)
    normalized_ground_truth = normalize_answer(ground_truth)

    ZERO_METRIC = (0, 0, 0)

    if normalized_prediction in ['yes', 'no', 'noanswer'] and normalized_prediction != normalized_ground_truth:
        return ZERO_METRIC
    if normalized_ground_truth in ['yes', 'no', 'noanswer'] and normalized_prediction != normalized_ground_truth:
        return ZERO_METRIC
    
    prediction_tokens = normalized_prediction.split()
    ground_truth_tokens = normalized_ground_truth.split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return ZERO_METRIC
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1, precision, recall

# ====================
# Gym Spaces (from ReAct-master)
# ====================

class textSpace(gym.spaces.Space):
    def contains(self, x) -> bool:
        """Return boolean specifying if x is a valid member of this space."""
        return isinstance(x, str)

# ====================
# Memory/Knowledge Base (Offline)
# ====================

class OfflineKnowledgeBase:
    """Local knowledge base - no external API calls."""
    def __init__(self):
        self.knowledge_base = {}
        self.search_count = 0
    
    def add_knowledge(self, topic, content):
        """Add knowledge about a topic."""
        self.knowledge_base[topic.lower()] = content
    
    def search(self, query):
        """Search local knowledge base."""
        self.search_count += 1
        query_lower = query.lower()
        
        # Exact match
        if query_lower in self.knowledge_base:
            return self.knowledge_base[query_lower]
        
        # Partial match
        for key in self.knowledge_base:
            if query_lower in key or key in query_lower:
                return self.knowledge_base[key]
        
        # Not found
        return None
    
    def lookup_in_text(self, keyword, text):
        """Find keyword in text (similar to lookup[])."""
        if not text:
            return []
        
        sentences = text.split('. ')
        matches = [s.strip() for s in sentences if keyword.lower() in s.lower() and len(s.strip()) > 5]
        return matches

# Initialize knowledge base
kb = OfflineKnowledgeBase()

# Pre-populate with common science facts
kb.add_knowledge("cell division", "Cell division is the process by which cells reproduce. There are three types: binary fission, mitosis, and meiosis.")
kb.add_knowledge("mitochondria", "Mitochondria are organelles found in eukaryotic cells. They are responsible for energy production through cellular respiration.")
kb.add_knowledge("photosynthesis", "Photosynthesis is the process by which plants convert light energy into chemical energy stored in glucose.")
kb.add_knowledge("dna", "DNA (deoxyribonucleic acid) is the molecule that carries genetic instructions for all known organisms.")
kb.add_knowledge("protein", "Proteins are large biomolecules composed of amino acids. They perform many functions including catalysis and structure.")
kb.add_knowledge("enzyme", "Enzymes are proteins that act as biological catalysts, speeding up chemical reactions in living organisms.")
kb.add_knowledge("electron", "An electron is a subatomic particle with negative electric charge. It is found in atoms and molecules.")
kb.add_knowledge("atom", "An atom is the smallest unit of matter that retains the properties of an element.")
kb.add_knowledge("molecule", "A molecule is a group of atoms bonded together.")

# ====================
# WikiEnv (Offline) - From ReAct-Master
# ====================

class WikiEnv(gym.Env):
    """Offline Wikipedia environment for ReAct (no external API calls)."""
    
    def __init__(self, knowledge_base=None):
        super().__init__()
        self.page = None
        self.obs = None
        self.lookup_keyword = None
        self.lookup_list = None
        self.lookup_cnt = None
        self.steps = 0
        self.answer = None
        self.observation_space = self.action_space = textSpace()
        self.search_time = 0
        self.num_searches = 0
        self.kb = knowledge_base if knowledge_base is not None else kb
        self.search_history = []
    
    def _get_obs(self):
        return self.obs
    
    def _get_info(self):
        return {"steps": self.steps, "answer": self.answer}
    
    def reset(self, seed=None, return_info=False, options=None):
        self.obs = ("Offline ReAct environment using local knowledge base.\n"
                    "Tools: search[query], lookup[keyword], finish[answer]\n")
        self.page = None
        self.lookup_keyword = None
        self.lookup_list = None
        self.lookup_cnt = None
        self.steps = 0
        self.answer = None
        self.search_history = []
        observation = self._get_obs()
        info = self._get_info()
        return (observation, info) if return_info else observation
    
    def construct_lookup_list(self, keyword):
        """Find all sentences containing keyword in current page."""
        if self.page is None:
            return []
        paragraphs = self.page.split("\n")
        paragraphs = [p.strip() for p in paragraphs if p.strip()]
        
        sentences = []
        for p in paragraphs:
            sentences += p.split('. ')
        sentences = [s.strip() + '.' for s in sentences if s.strip()]
        
        parts = [p for p in sentences if keyword.lower() in p.lower()]
        return parts
    
    @staticmethod
    def get_page_obs(page, max_sentences=5):
        """Get observation from page (first N sentences)."""
        paragraphs = page.split("\n")
        paragraphs = [p.strip() for p in paragraphs if p.strip()]
        
        sentences = []
        for p in paragraphs:
            sentences += p.split('. ')
        sentences = [s.strip() + '.' for s in sentences if s.strip()]
        return ' '.join(sentences[:max_sentences])
    
    def search_local(self, entity):
        """Search local knowledge base (no external API)."""
        old_time = time.time()
        self.num_searches += 1
        
        # Search in knowledge base
        result = self.kb.search(entity)
        
        if result:
            self.page = result
            self.obs = self.get_page_obs(self.page)
            self.lookup_keyword = self.lookup_list = self.lookup_cnt = None
        else:
            self.obs = f"Could not find '{entity}' in local knowledge base. Try a different search term."
            self.result_titles = []
        
        self.search_time += time.time() - old_time
        self.search_history.append((entity, bool(result)))
    
    def step(self, action):
        reward = 0
        done = False
        action = action.strip()
        
        if self.answer is not None:
            done = True
            return self.obs, reward, done, self._get_info()
        
        if action.startswith("search[") and action.endswith("]"):
            entity = action[len("search["):-1]
            self.search_local(entity)
        elif action.startswith("lookup[") and action.endswith("]"):
            keyword = action[len("lookup["):-1]
            if self.lookup_keyword != keyword:
                self.lookup_keyword = keyword
                self.lookup_list = self.construct_lookup_list(keyword)
                self.lookup_cnt = 0
            if self.lookup_cnt >= len(self.lookup_list):
                self.obs = "No more results.\n"
            else:
                self.obs = f"(Result {self.lookup_cnt + 1} / {len(self.lookup_list)}) " + self.lookup_list[self.lookup_cnt]
                self.lookup_cnt += 1
        elif action.startswith("finish[") and action.endswith("]"):
            answer = action[len("finish["):-1]
            self.answer = answer
            done = True
            self.obs = f"Episode finished, answer = {answer}\n"
        elif action.startswith("think[") and action.endswith("]"):
            self.obs = "Nice thought."
        else:
            self.obs = "Invalid action: {}".format(action)
        
        self.steps += 1
        return self.obs, reward, done, self._get_info()
    
    def get_time_info(self):
        speed = self.search_time / self.num_searches if self.num_searches else 0
        return {
            "call_speed": speed,
            "call_time": self.search_time,
            "num_calls": self.num_searches,
        }

# ====================
# Environment Wrappers (from ReAct-Master)
# ====================

class HistoryWrapper(gym.ObservationWrapper):
    """Wraps observations to include full history."""
    def __init__(self, env, obs_format="obs", prompt=None):
        super().__init__(env)
        assert obs_format in ["obs", "history"]
        self.obs_format = obs_format
        self.prompt = prompt if prompt is not None else ""

    def observation(self, obs):
        if self.obs_format == "obs":
            return obs
        elif self.obs_format == "history":
            if hasattr(self.env, 'traj'):
                observation = self.env.traj["observations"][0] + "\n"
                for i, (o, a) in enumerate(zip(self.env.traj["observations"][1:], self.env.traj["actions"]), 1):
                    observation += f"Action {i}: {a}\nObservation {i}: {o}\n\n"
                return self.prompt + observation
        return obs

class LoggingWrapper(gym.Wrapper):
    """Logs all interactions (from ReAct-Master)."""
    def __init__(self, env, folder="trajs", file_id=None):
        super().__init__(env)
        self.trajs = []
        self.traj = {"observations": [], "actions": []}
        self.folder = folder
        self.file_id = np.random.randint(0, 10000000) if file_id is None else file_id
        self.file_path = f"{self.folder}/{self.file_id}.json"
        os.makedirs(self.folder, exist_ok=True)

    def reset(self, seed=None, return_info=False, options=None, idx=None):
        output = self.env.reset(seed=seed, return_info=return_info, options=options)
        observation = output[0] if return_info else output
        self.traj = {"observations": [observation], "actions": []}
        return output

    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        self.traj["observations"].append(obs)
        self.traj["actions"].append(action)
        if done:
            self.traj.update(info)
        return obs, reward, done, info

    def update_record(self):
        if len(self.traj) > 0:
            self.trajs.append(self.traj)
            self.traj = {"observations": [], "actions": []}
    
    def write(self):
        self.update_record()
        with open(self.file_path, "w") as f:
            json.dump(self.trajs, f)
        return self.file_path
    
    def close(self):
        return self.write()

print("✓ ReAct-Master complete implementation loaded (OFFLINE - No external API calls)")

## ReAct Solver

In [ ]:
def llm_generate(prompt: str, temperature: float = 0.3, max_tokens: int = 500) -> str:
    """Generate from DeepSeek using HuggingFace transformers."""
    try:
        # Format prompt for instruction-tuned model
        formatted_prompt = f"User: {prompt}\n\nAssistant:"
        
        # Tokenize
        inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=2048)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract assistant's response
        if "Assistant:" in response:
            response = response.split("Assistant:")[-1].strip()
        
        return response
        
    except Exception as e:
        print(f"⚠️ LLM error: {e}")
        return ""

def parse_action(text: str) -> tuple:
    """
    Parse action from LLM output (ReAct-master style).
    Returns: (action_type, parameter)
    """
    patterns = [
        (r'search\[(.+?)\]', 'search'),
        (r'lookup\[(.+?)\]', 'lookup'),
        (r'finish\[([A-D])\]', 'finish'),
        (r'think\[(.+?)\]', 'think'),
    ]
    t = text.lower()
    for pattern, action_type in patterns:
        m = re.search(pattern, t, re.IGNORECASE | re.DOTALL)
        if m:
            param = m.group(1).strip()
            # For finish, ensure it's A-D
            if action_type == 'finish' and param not in ['A', 'B', 'C', 'D']:
                param = param.upper()
                if param not in ['A', 'B', 'C', 'D']:
                    continue
            return action_type, param
    return None, None

def extract_final_answer(text: str) -> str:
    """Extract ONLY the answer letter A/B/C/D from text (multiple patterns)."""
    text_clean = re.sub(r'\s+', '', text.upper())
    
    patterns = [
        r'FINISH\[([A-D])\]',
        r'FINALANSWER\s*:\s*([A-D])',
        r'ANSWER\s*:\s*([A-D])',
        r'[^A-D]([A-D])[^A-D]',  # Isolated letter
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text_clean)
        if match:
            return match.group(1)
    
    # Last resort: find any A-D at end
    match = re.search(r'([A-D])', text[-50:].upper())
    if match:
        return match.group(1)
    
    return ""

def react_solve_question(example: dict, max_steps: int = REACT_MAX_STEPS) -> dict:
    """
    ReAct question solver with offline environment (from ReAct-master).
    Uses WikiEnv for local tool execution - NO external API calls.
    """
    question = example.get('question', '')
    correct_answer = example.get('correct_answer', '')
    index = example.get('index', -1)
    
    # Initialize offline environment
    env = WikiEnv(knowledge_base=kb)
    obs = env.reset()
    
    steps = []
    final_answer = "?"
    
    # Build ReAct prompt
    prompt = f"""You are an expert solving PhD-level science questions.
Use the ReAct (Reasoning + Acting) approach with these tools:
- search[topic]: Search local knowledge for a topic
- lookup[keyword]: Find keyword in current page
- finish[X]: Return final answer (A, B, C, or D)

Question:
{question}

Format each step as:
Thought: [your reasoning]
Action: [tool call]

Begin solving:
"""
    
    try:
        for step_num in range(1, max_steps + 1):
            # Generate thought and action
            step_prompt = prompt + f"\nThought {step_num}: "
            output = llm_generate(step_prompt, temperature=0.3, max_tokens=300)
            
            # Parse thought and action
            lines = output.split('\n')
            thought = ""
            action_text = ""
            
            # Find where action starts
            action_start_idx = len(lines)
            for i, line in enumerate(lines):
                if 'action' in line.lower() and ':' in line:
                    action_start_idx = i
                    break
            
            # Thought is everything before action
            thought = '\n'.join(lines[:action_start_idx]).strip()
            
            # Action is at or after action_start_idx
            if action_start_idx < len(lines):
                action_line = lines[action_start_idx]
                if ':' in action_line:
                    action_text = action_line.split(':', 1)[1].strip()
                else:
                    action_text = action_line.strip()
            
            if not action_text:
                action_text = output.strip()
            
            # Parse action command
            action_type, parameter = parse_action(action_text)
            
            # Execute action in offline environment
            if action_type == 'search':
                obs, reward, done, info = env.step(f"search[{parameter}]")
            elif action_type == 'lookup':
                obs, reward, done, info = env.step(f"lookup[{parameter}]")
            elif action_type == 'finish':
                final_answer = parameter if parameter in ['A','B','C','D'] else "?"
                obs, reward, done, info = env.step(f"finish[{final_answer}]")
                done = True
            else:
                obs = "Invalid action. Use: search[...], lookup[...], or finish[A/B/C/D]"
                done = False
            
            # Record step
            steps.append({
                "iteration": step_num,
                "thought": thought[:150],
                "action": f"{action_type}[{parameter}]" if action_type else action_text[:100],
                "observation": obs[:200] if obs else "No result"
            })
            
            # Update prompt with history
            prompt += f"\nThought {step_num}: {thought[:80]}\nAction {step_num}: {action_text[:80]}\nObservation {step_num}: {obs[:120]}"
            
            if done:
                break
                
    except Exception as e:
        steps.append({
            "iteration": len(steps) + 1,
            "thought": "Error during reasoning",
            "action": "error",
            "observation": str(e)[:100]
        })
    
    # Extract final answer if not set
    if not final_answer or final_answer not in ['A','B','C','D']:
        final_answer = extract_final_answer(prompt)
    
    if not final_answer or final_answer not in ['A','B','C','D']:
        final_answer = "?"
    
    # Build complete trace
    trace_lines = ["=== ReAct Reasoning Trace (Offline) ==="]
    for s in steps:
        trace_lines.append(f"Thought {s['iteration']}: {s['thought']}")
        trace_lines.append(f"Action {s['iteration']}: {s['action']}")
        trace_lines.append(f"Observation {s['iteration']}: {s['observation']}")
    trace_lines.append(f"\nFinal Answer: {final_answer}")
    trace = "\n".join(trace_lines)
    
    return {
        "index": index,
        "question": question[:200],
        "correct_answer": correct_answer,
        "react_answer": final_answer,
        "is_correct": final_answer == correct_answer,
        "react_trace": trace,
        "steps": steps,
    }

print("✓ ReAct solver with offline environment ready (from ReAct-master - NO external API)")

## Batch Evaluation with Per-Question Checkpoints

In [ ]:
# Prepare data
formatted_data = []
for idx, ex in enumerate(gpqa_dataset):
    q = ex.get('question', '')
    ans = (ex.get('answer','') or '').strip().upper()
    if len(ans) > 1:
        m = re.search(r'([A-D])', ans)
        if m:
            ans = m.group(1)
    formatted_data.append({
        'index': idx,
        'question': q,
        'correct_answer': ans
    })
print(f"Prepared {len(formatted_data)} examples")

# Load checkpoint
checkpoint_data = {"evaluated_indices": set(), "accumulated_results": []}
if GPQA_CHECKPOINT_PATH.exists():
    try:
        with open(GPQA_CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
            saved = json.load(f)
            checkpoint_data['evaluated_indices'] = set(saved.get('evaluated_indices', []))
            checkpoint_data['accumulated_results'] = saved.get('accumulated_results', [])
        print(f"✓ Checkpoint: {len(checkpoint_data['evaluated_indices'])} already evaluated")
    except:
        print("⚠️ Checkpoint corrupted, starting fresh")

# Compute remaining
all_indices = set(range(len(formatted_data)))
remaining = sorted(all_indices - checkpoint_data['evaluated_indices'])

if not remaining:
    print("✓ All examples already evaluated!")
    results = checkpoint_data['accumulated_results']
else:
    batch_indices = remaining[:BATCH_SIZE]
    print(f"\nEvaluating batch: indices {batch_indices[0]}-{batch_indices[-1]}")
    print(f"Progress: {len(checkpoint_data['evaluated_indices'])}/{len(formatted_data)} done")
    
    results = []
    for idx in tqdm(batch_indices, desc="ReAct+DeepSeek"):
        try:
            result = react_solve_question(formatted_data[idx])
            results.append(result)
            checkpoint_data['evaluated_indices'].add(idx)
            checkpoint_data['accumulated_results'].append(result)
            
            # Save incrementally
            with open(GPQA_OUTPUT_PATH, 'a', encoding='utf-8') as f:
                json.dump({
                    "index": result['index'],
                    "question": result['question'],
                    "answer": result['react_answer'],
                    "correct_answer": result['correct_answer'],
                    "is_correct": result['is_correct'],
                    "react_trace": result['react_trace'],
                    "timestamp": datetime.now().isoformat()
                }, f, ensure_ascii=False)
                f.write("\n")
            
            with open(GPQA_TRACES_PATH, 'a', encoding='utf-8') as f:
                json.dump(result, f, ensure_ascii=False)
                f.write("\n")
            
            # Save checkpoint after EACH question
            with open(GPQA_CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
                json.dump({
                    'evaluated_indices': sorted(list(checkpoint_data['evaluated_indices'])),
                    'accumulated_results': checkpoint_data['accumulated_results'][-50:],
                    'timestamp': datetime.now().isoformat()
                }, f, ensure_ascii=False, indent=2)
        
        except Exception as e:
            print(f"⚠️ Error on index {idx}: {str(e)[:100]}")
            continue
    
    if results:
        correct_count = sum(1 for r in results if r['is_correct'])
        batch_accuracy = correct_count / len(results) * 100
        print(f"\n✓ Batch complete: {correct_count}/{len(results)} correct ({batch_accuracy:.1f}%)")
    else:
        results = []
        print("⚠️ No results generated")

print(f"Total: {len(checkpoint_data['evaluated_indices'])}/{len(formatted_data)}")

## Metrics & Analysis

In [ ]:
# Calculate metrics
results_df = pd.DataFrame(results)
correct_results = results_df[results_df['is_correct'] == True]
incorrect_results = results_df[results_df['is_correct'] == False]

batch_correct = len(correct_results)
total_batch = len(results_df)
batch_accuracy = batch_correct / total_batch * 100 if total_batch else 0

print("\n" + "="*60)
print("BATCH ANALYSIS")
print("="*60)
print(f"Correct: {batch_correct}/{total_batch} ({batch_accuracy:.1f}%)")

if len(incorrect_results) > 0:
    print("\nSample incorrect (first 3):")
    for _, row in incorrect_results.head(3).iterrows():
        print(f"  Q: {row['question'][:90]}...")
        print(f"  Model: {row['react_answer']} | Gold: {row['correct_answer']}")

# Cumulative metrics
all_eval = len(checkpoint_data['evaluated_indices'])
cumulative_stats = {'total_all_batches': all_eval, 'correct_all_batches': 0, 'batches_completed': 0}
if GPQA_CUMULATIVE_PATH.exists():
    try:
        with open(GPQA_CUMULATIVE_PATH, 'r') as f:
            cumulative_stats = json.load(f)
    except:
        pass

cumulative_stats['total_all_batches'] = all_eval
cumulative_stats['correct_all_batches'] += batch_correct
cumulative_stats['batches_completed'] = (all_eval // BATCH_SIZE) + (1 if all_eval % BATCH_SIZE else 0)
cumulative_stats['last_batch_accuracy'] = batch_accuracy
cumulative_stats['last_updated'] = datetime.now().isoformat()

cumulative_accuracy = (cumulative_stats['correct_all_batches'] / all_eval * 100) if all_eval else 0

with open(GPQA_CUMULATIVE_PATH, 'w') as f:
    json.dump(cumulative_stats, f, indent=2)

metrics = {
    "dataset": "GPQA-Diamond",
    "model": MODEL_NAME,
    "method": "ReAct with Tool Use",
    "current_batch": {
        "batch_size": len(results),
        "correct": batch_correct,
        "accuracy_pct": f"{batch_accuracy:.2f}%"
    },
    "cumulative": {
        "total_evaluated": all_eval,
        "total_correct": cumulative_stats['correct_all_batches'],
        "accuracy_pct": f"{cumulative_accuracy:.2f}%",
        "batches_completed": cumulative_stats['batches_completed'],
        "remaining": len(formatted_data) - all_eval
    },
    "output_files": {
        "results_jsonl": str(GPQA_OUTPUT_PATH),
        "traces_jsonl": str(GPQA_TRACES_PATH),
        "metrics_json": str(GPQA_METRICS_PATH),
        "checkpoint_json": str(GPQA_CHECKPOINT_PATH)
    },
    "timestamp": datetime.now().isoformat()
}

with open(GPQA_METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)

print("\n" + "="*60)
print("METRICS SUMMARY")
print("="*60)
print(json.dumps(metrics, indent=2))
print("\n" + "="*60)
print("FILES SAVED")
print("="*60)
for k, v in metrics["output_files"].items():
    print(f"  {k}: {v}")
print("="*60)

---
## 📋 Next Steps

**To continue processing:**
1. Re-run cell 9 (Batch Evaluation) to process next 10 questions
2. Checkpoints are saved after **EACH question** - you won't lose progress!
3. Each run saves:
   - `gpqa_react_results.jsonl` - Results summary
   - `gpqa_react_detailed_traces.jsonl` - Full reasoning traces
   - `gpqa_react_checkpoint.json` - Progress checkpoint (updated per question)
   - `gpqa_metrics.json` - Accuracy metrics

**Download outputs:**
- Click folder icon (left sidebar) → `outputs/` → Download files before session ends

**Estimated time:** ~20 batches to complete all 198 questions (2-4 hours on GPU)